# 01. Data and Tables 1–2

All numbers are computed from the Dryad files (`data/`). Run `python scripts/run_pipeline.py` first (processed tables are otherwise built on demand).

In [1]:
import numpy as np
import pandas as pd
from tbopt import config, data
AGES = config.AGE_GROUPS_40

## Table 1 — national totals (province tables)

In [2]:
table1 = data.sido_summary_table()
table1

,N,D,H
year,,,
2014,34939,2136,330
2015,32182,2018,334
2016,30892,2020,338
2017,28161,1678,343
2018,26433,1657,345
2019,23830,1492,353
2020,19542,1222,361
2021,18335,1324,363
2022,16932,1223,373


In [3]:
print(data.table_to_latex(table1))

2014 & 34 939 & 2 136 & 330 \\
2015 & 32 182 & 2 018 & 334 \\
2016 & 30 892 & 2 020 & 338 \\
2017 & 28 161 & 1 678 & 343 \\
2018 & 26 433 & 1 657 & 345 \\
2019 & 23 830 & 1 492 & 353 \\
2020 & 19 542 & 1 222 & 361 \\
2021 & 18 335 & 1 324 & 363 \\
2022 & 16 932 & 1 223 & 373 \\


## District filter

District age-specific deaths are reconstructed as $D_{s,t} = d_{i,t} D_s$ with the province fraction $d_{i,t}$. A district is kept if

1. it has at least one hospital ($H_s > 0$),
2. at least one TB death of any age ($D_s > 0$), and
3. its province reports deaths **and** new cases in every 40+ age group ($D_{s,t} > 0$, $N_{s,t} > 0$ for $t=1..5$).

Condition 3 removes all districts of a province with no deaths in some 40+ age group.

In [4]:
steps = []
for y in config.YEARS:
    df = data.reconstruct_age(data.load_district(y))
    m1 = df["h"] > 0
    m2 = m1 & (df["D"] > 0)
    m3 = m2 & np.all([df[f"phi_{a}"] > data.PHI_MIN for a in AGES], axis=0)
    prov = df.drop_duplicates("sido").set_index("sido")
    missing = {s: [a for a in AGES if prov.loc[s, f"d{a}"] == 0] for s in prov.index}
    steps.append({"year": y, "all": len(df), "H>0": int(m1.sum()), "+D>0": int(m2.sum()),
                  "+40+ ages": int(m3.sum()),
                  "provinces removed (age groups without deaths)": {s: v for s, v in missing.items() if v}})
pd.DataFrame(steps).set_index("year")

,all,H>0,+D>0,+40+ ages,provinces removed (age groups without deaths)
year,,,,,
2014,228,145,143,143,{}
2015,228,148,147,142,{'Daejeon': ['40_49']}
2016,228,148,147,136,"{'Daejeon': ['40_49'], 'Ulsan': ['40_49'], 'Je..."
2017,228,148,147,145,{'Jeju': ['70_79']}
2018,228,148,148,146,"{'Jeju': ['40_49', '60_69']}"
2019,228,149,147,125,"{'Daejeon': ['40_49'], 'Ulsan': ['40_49'], 'Gy..."
2020,228,149,147,121,"{'Gwangju': ['50_59'], 'Daejeon': ['40_49'], '..."
2021,228,149,145,112,"{'Daejeon': ['40_49'], 'Ulsan': ['40_49'], 'Ch..."
2022,228,149,144,124,"{'Daegu': ['40_49'], 'Daejeon': ['50_59'], 'Ul..."


## Table 2 — districts used in the analysis

`D` = all-age deaths, `D_40` = reconstructed 40+ deaths (equals $E$ of the current allocation).

In [5]:
table2 = data.summary_table()
table2

,S,N,D,D_40,H
year,,,,,
2014,143,29886,1718,1688.717602,328
2015,142,26849,1635,1604.908918,322
2016,136,24876,1558,1539.287206,316
2017,145,23727,1351,1339.078405,335
2018,146,22159,1310,1302.847890,340
2019,125,17584,1088,1080.447563,306
2020,121,14978,878,864.814897,291
2021,112,12383,838,832.935397,264
2022,124,11841,886,884.509091,314


In [6]:
print(data.table_to_latex(table2))

2014 & 143 & 29 886 & 1 718 & 1 688.7 & 328 \\
2015 & 142 & 26 849 & 1 635 & 1 604.9 & 322 \\
2016 & 136 & 24 876 & 1 558 & 1 539.3 & 316 \\
2017 & 145 & 23 727 & 1 351 & 1 339.1 & 335 \\
2018 & 146 & 22 159 & 1 310 & 1 302.8 & 340 \\
2019 & 125 & 17 584 & 1 088 & 1 080.4 & 306 \\
2020 & 121 & 14 978 & 878 & 864.8 & 291 \\
2021 & 112 & 12 383 & 838 & 832.9 & 264 \\
2022 & 124 & 11 841 & 886 & 884.5 & 314 \\


## Comparison with the submitted manuscript

The submitted Table 2 was computed from an older version of the processed data; only 2014 and 2022 agree in S, D, H.

In [7]:
paper1 = pd.DataFrame([(34869, 2136, 330), (32182, 2018, 335), (30892, 2020, 339), (28161, 1678, 344),
                       (26433, 1657, 346), (23821, 1492, 354), (19933, 1222, 360), (18335, 1324, 364),
                       (16264, 1223, 374)], index=config.YEARS, columns=["N", "D", "H"])
paper2 = pd.DataFrame([(143, 32322, 1718, 328), (137, 21683, 1296, 312), (132, 20207, 1255, 306),
                       (143, 19427, 1110, 330), (142, 18145, 1157, 334), (119, 14318, 901, 285),
                       (116, 12079, 736, 276), (113, 10900, 749, 262), (124, 11789, 886, 314)],
                      index=config.YEARS, columns=["S", "N", "D", "H"])
cmp1 = pd.concat({"now": table1, "submitted": paper1}, axis=1).swaplevel(axis=1).sort_index(axis=1)
cmp2 = pd.concat({"now": table2[paper2.columns], "submitted": paper2}, axis=1).swaplevel(axis=1).sort_index(axis=1)
display(cmp1, cmp2)

D              H                N          
       now submitted  now submitted    now submitted
2014  2136      2136  330       330  34939     34869
2015  2018      2018  334       335  32182     32182
2016  2020      2020  338       339  30892     30892
2017  1678      1678  343       344  28161     28161
2018  1657      1657  345       346  26433     26433
2019  1492      1492  353       354  23830     23821
2020  1222      1222  361       360  19542     19933
2021  1324      1324  363       364  18335     18335
2022  1223      1223  373       374  16932     16264

D              H                N              S          
       now submitted  now submitted    now submitted  now submitted
2014  1718      1718  328       328  29886     32322  143       143
2015  1635      1296  322       312  26849     21683  142       137
2016  1558      1255  316       306  24876     20207  136       132
2017  1351      1110  335       330  23727     19427  145       143
2018  1310      1157  340       334  22159     18145  146       142
2019  1088       901  306       285  17584     14318  125       119
2020   878       736  291       276  14978     12079  121       116
2021   838       749  264       262  12383     10900  112       113
2022   886       886  314       314  11841     11789  124       124

## Check: processed data equals the Dryad filtered files

In [8]:
for y in config.YEARS:
    p, f = data.load_processed(y), data.load_district(y, filtered=True)
    assert len(p) == len(f) and all(p[c].sum() == f[c].sum() for c in ["N", "D", "h"]), y
print("OK")

OK


## Check: province age fractions vs. province count tables

The district files carry province fractions `n_t`, `d_t`. Recomputing them from the Dryad province count tables gives the same values except for the cases below (an internal inconsistency of the dataset; the model uses the fractions in the district files).

In [9]:
rows = []
for y in config.YEARS:
    for kind, used, recomputed in zip("nd", data.sido_age_fractions(y), data.sido_age_fractions_from_counts(y)):
        diff = (used - recomputed.loc[used.index]).abs()
        for s in diff.index[(diff > 1e-6).any(axis=1)]:
            rows.append({"year": y, "fraction": kind, "sido": s, "max_abs_diff": float(diff.loc[s].max())})
pd.DataFrame(rows)

,year,fraction,sido,max_abs_diff
0,2014,n,Seoul,0.001271
1,2014,n,Daegu,0.026968
2,2019,n,Jeju,0.027106
3,2019,d,Chungnam,0.014493
4,2020,n,Chungnam,0.065141
5,2020,n,Jeonbuk,0.096373
6,2022,n,Seoul,0.008838
